In [11]:
%load_ext autoreload 
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Alter data prep to be exactly like the Deep Triangle paper. Each Dev period will have a target SEQUENCE and an input SEQUENCE.
<br><br> Later down the line we can choose it to be either an Autoregressive MODEL or a SEQ2SEQ model. 


In [12]:

from rnn_reserving.data_import import read_local_raw_data, process_data, split_data
import pandas as pd
from torch.nn.utils.rnn import pad_sequence

In [13]:
df_cas = read_local_raw_data()
    
df_cas = process_data(df_cas)
df_cas = split_data(df_cas)

In [14]:
df_test = df_cas[df_cas['GRCODE'] == 43][
    ['AccidentYear',
    'DevelopmentLag',
    'incurred_loss_ratio',
    'paid_loss_ratio',
    'case_loss_ratio',
    'calendar_year',
    'GRCODE_mapped',
    'bucket'
    ]
].copy()

The below shows train in blue, val in green and test in red! Yay! 


In [7]:
ilr_triangle = (
    df_test
    .pivot(
        index="AccidentYear",
        columns="DevelopmentLag",
        values="paid_loss_ratio"
    )
    .sort_index()
)

bucket_triangle = (
    df_test
    .pivot(
        index="AccidentYear",
        columns="DevelopmentLag",
        values="bucket"
    )
    .sort_index()
)

def style_bucket(val):
    if val == "train":
        return "background-color: #cce5ff; color: #003366;"
    if val == "validation":
        return "background-color: #d4edda; color: #155724;"
    if val == "test":
        return "background-color: #f8d7da; color: #721c24;"
    return ""


styled = (
    ilr_triangle
    .style
    .apply(
        lambda _: bucket_triangle.applymap(style_bucket),
        axis=None
    )
)

styled


C:\Users\TobyCook\AppData\Local\Temp\ipykernel_22652\2490517049.py:35: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  lambda _: bucket_triangle.applymap(style_bucket),


DevelopmentLag,1,2,3,4,5,6,7,8,9,10
AccidentYear,,,,,,,,,,
1988,0.148603,0.372067,0.481564,0.636872,0.687151,0.687151,0.687151,0.686034,0.686034,0.686034
1989,0.274141,0.512474,0.694159,0.756971,0.810977,0.870561,0.862929,0.874083,0.874083,0.862342
1990,0.344710,0.825947,1.168280,1.373238,1.459501,1.484632,1.488029,1.487859,1.482255,1.494651
1991,0.270317,0.686785,0.901037,0.992374,1.031995,1.076978,1.090801,1.108317,1.098308,1.100095
1992,0.273592,0.580931,0.812566,0.911636,0.959128,0.981083,0.983506,1.001194,0.999928,1.000543
1993,0.237435,0.566338,0.760772,0.856025,0.889322,0.908048,0.910382,0.923530,0.926063,0.925807
1994,0.294918,0.631898,0.794907,0.899847,0.941509,0.974534,0.986527,0.988241,0.990189,0.990846
1995,0.282118,0.546138,0.665078,0.730542,0.773499,0.801088,0.814126,0.815680,0.820676,0.821768
1996,0.268576,0.499606,0.602549,0.672335,0.715954,0.739180,0.750836,0.752094,0.752520,0.752797


In [8]:
# We do need to handle the val data having a full history for input but only a partial for target though..

In [9]:
feature_cols = ['paid_loss_ratio']

In [27]:
pad_dims = (0, 0) * (3 - 1) + (0, 1)
pad_dims
# padded.append(F.pad(t, pad_dims, value=pad_value))

(0, 0, 0, 0, 0, 1)

In [ ]:
import numpy as np 
from collections import defaultdict

def build_sequences(df, feature_cols):
    data = {
        "train": defaultdict(list),
        "validation": defaultdict(list),
        "test": defaultdict(list),
    }

    for (ay, cc), group in df.groupby(["AccidentYear", "GRCODE_mapped"]):
        print('Processing AY:', ay, 'GRCODE_mapped:', cc)
        group = group.sort_values("DevelopmentLag").reset_index(drop=True)
        
        for i in range(len(group) - 1):
            target_split = group.loc[i+1, "bucket"]
            
            input_seq = group.loc[:i, feature_cols].values
            
            future_mask = (
                (group.index > i) &
                (group["bucket"] == target_split)
            )
            
            target_seq = group.loc[future_mask, feature_cols[0]].values
            
            data[target_split]["inputs"].append(input_seq)
            data[target_split]["targets"].append(target_seq)
            data[target_split]["lengths"].append(i + 1)  # actual length before padding
            data[target_split]["ids"].append(
                (ay, cc, group.loc[i, "DevelopmentLag"], target_split)
            )

    return data


data = build_sequences(df_test, feature_cols)


Processing AY: 1988 GRCODE_mapped: 0
Processing AY: 1989 GRCODE_mapped: 0
Processing AY: 1990 GRCODE_mapped: 0
Processing AY: 1991 GRCODE_mapped: 0
Processing AY: 1992 GRCODE_mapped: 0
Processing AY: 1993 GRCODE_mapped: 0
Processing AY: 1994 GRCODE_mapped: 0
Processing AY: 1995 GRCODE_mapped: 0
Processing AY: 1996 GRCODE_mapped: 0
Processing AY: 1997 GRCODE_mapped: 0


In [42]:
data['train']['inputs']

[array([[0.14860335]]),
 array([[0.14860335],
        [0.37206704]]),
 array([[0.14860335],
        [0.37206704],
        [0.48156425]]),
 array([[0.14860335],
        [0.37206704],
        [0.48156425],
        [0.63687151]]),
 array([[0.14860335],
        [0.37206704],
        [0.48156425],
        [0.63687151],
        [0.68715084]]),
 array([[0.14860335],
        [0.37206704],
        [0.48156425],
        [0.63687151],
        [0.68715084],
        [0.68715084]]),
 array([[0.14860335],
        [0.37206704],
        [0.48156425],
        [0.63687151],
        [0.68715084],
        [0.68715084],
        [0.68715084]]),
 array([[0.27414147]]),
 array([[0.27414147],
        [0.51247432]]),
 array([[0.27414147],
        [0.51247432],
        [0.69415908]]),
 array([[0.27414147],
        [0.51247432],
        [0.69415908],
        [0.75697094]]),
 array([[0.27414147],
        [0.51247432],
        [0.69415908],
        [0.75697094],
        [0.8109774 ]]),
 array([[0.27414147],
        

In [43]:
for key in data.keys():
    l = max(data[key]['lengths'])
    print(f"{key}: {len(data[key]['inputs'])} sequences, max length: {l}")

train: 28 sequences, max length: 7
validation: 17 sequences, max length: 9
test: 45 sequences, max length: 9


In [44]:
## OK great 2026-02-15:
# Finally happy with my little sequences.. 
# Next steps get into dataloader. Maybe use a mask, rather than padding now.. Or could chuck in the padding... 
# Then we can run a seq2seq ... really not much work left now ... 

In [45]:
from torch.utils.data import Dataset, DataLoader
import torch
from typing import Tuple, List, Dict, Any
import logging 

logger = logging.getLogger(__name__)

In [46]:
class InsuranceForecastDataset(Dataset):
    """
    Dataset for insurance time series forecasting.
    
    Input: periods 0..(seq_len-2)
    Target: periods 1..(seq_len-1) (next-step prediction)
    
    Args:
        input_seqs: List of input sequences (variable length)
        target_seqs: List of target sequences (variable length)
        lengths: Original sequence lengths before padding
        ids: Sequence identifiers
    """
    
    def __init__(
        self,
        input_seqs: List[np.ndarray],
        target_seqs: List[np.ndarray],
        lengths: List[int],
        ids: List[Any],
    ):
        # Validation
        assert len(input_seqs) == len(target_seqs) == len(lengths) == len(ids), \
            "All inputs must have the same length"
        
        self.n_samples = len(input_seqs)

        logger.info("Caching dataset in memory...")
        self.input_seqs = [torch.FloatTensor(seq) for seq in input_seqs]
        self.target_seqs = [torch.FloatTensor(seq) for seq in target_seqs]

        self.lengths = lengths
        self.ids = ids

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        return (
            torch.FloatTensor(self.input_seqs[idx]),
            self.lengths[idx],
            torch.FloatTensor(self.target_seqs[idx]),
            self.ids[idx]
        )

In [47]:
train_data_set = InsuranceForecastDataset(
    input_seqs=data['train']['inputs'],
    target_seqs=data['train']['targets'],
    lengths=data['train']['lengths'],
    ids=data['train']['ids']    
)

In [48]:

def collate_fn(
    batch,
    pad_value: float = 0.0,
):
    """ Pad both inputs and outputs to max of batch length """
    inputs, lengths, targets, ids = zip(*batch)

    padded_inputs = pad_sequence(inputs, batch_first=True, padding_value=pad_value)
    padded_targets = pad_sequence(targets, batch_first=True, padding_value=pad_value)
    lengths_tensor = torch.LongTensor(lengths)

    batch = {
        'inputs': padded_inputs,
        'lengths': lengths_tensor,
        'targets': padded_targets,
        'ids': ids
    }
    return batch

In [53]:
train_loader = DataLoader(train_data_set, batch_size=32, shuffle=False, collate_fn=collate_fn)